# M2-CL trên Colab — `vlcs` / resnet50 (GPU riêng)
Chạy ĐỘC LẬP 1 cặp dataset+backbone trên 1 Colab runtime/GPU riêng. Mở nhiều notebook = nhiều GPU song song thật.
**Data** để ở ổ LOCAL `/content` (nhanh, ổn định — KHÔNG giải nén lên Drive). **Output + log + checkpoint** lưu trên Drive → ngắt session vẫn không mất, Run all lại là train tiếp (`--skip_existing`). Data tải lại mỗi session vài phút.
Đủ 12 method = 10 baseline (erm,rsc,mixup,coral,mmd,sagnet,selfreg,arm,eqrm,sagm) + m2 + m2cl.
**W&B**: điền key vào cell 4b (key để TRỐNG trong file). Dùng cá nhân — ĐỪNG commit/up file ĐÃ điền key lên GitHub public.
Dùng: Runtime → Change runtime type → **A100 GPU** → Run all. (Batch tự dò = 128 = batch paper; A100 40GB chạy đúng paper + nhanh.)

In [ ]:
# 1) GPU + mount Drive
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2) Clone code + cài deps thiếu (giữ torch sẵn của Colab)
import os
WORK = '/content/drive/MyDrive/cv20252'
REPO = '/content/Computer-Vision-20252'
os.makedirs(WORK, exist_ok=True)
if not os.path.isdir(REPO):
    !git clone https://github.com/Nituv05/Computer-Vision-20252.git {REPO}
%cd {REPO}
!git pull -q
!pip install -q 'gdown>=5.2,<6.0' 'huggingface_hub[hf_xet]>=0.36' 'pyarrow>=14.0' 'wandb>=0.17'
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0))

In [ ]:
# 3) Cấu hình cho job NÀY (data + output RIÊNG theo dataset+backbone, không đụng notebook khác)
DS = 'vlcs'
BB = 'resnet50'
BBTAG = 'r50'
DATA_ROOT = f'/content/data_{DS}'              # DATA ở ổ LOCAL /content (nhanh, ổn định). KHÔNG để trên Drive!
SAVE_DIR  = f'{WORK}/outputs/{DS}_{BBTAG}'    # output trên Drive, RIÊNG theo backbone
import time; LOG = f'{WORK}/logs/{DS}_{BBTAG}_' + time.strftime('%Y%m%d_%H%M%S') + '.log'
os.makedirs(f'{WORK}/logs', exist_ok=True)
free_mb = torch.cuda.mem_get_info()[0] // 1024**2
BATCH_SIZE = 128 if free_mb >= 24000 else 64 if free_mb >= 20000 else 48 if free_mb >= 15000 else 32
print(f'DS={DS} backbone={BB} batch_size={BATCH_SIZE} log={LOG}')

In [ ]:
# 4) Tải dataset NÀY vào ổ LOCAL + kiểm tra layout.
#    Để data trên /content (không phải Drive): tránh lỗi giải nén 15k+ file qua Drive FUSE,
#    và train đọc ảnh nhanh hơn NHIỀU. Mỗi session tải lại vài phút là đáng.
if not os.path.isdir(f'{DATA_ROOT}/{DS}'):
    !python download_data.py --data_root {DATA_ROOT} --datasets {DS}
!python check_data.py --data_root {DATA_ROOT} --dataset {DS}

In [ ]:
# 4b) Điền W&B API key của bạn vào đây (dùng cá nhân; ĐỪNG up file đã điền lên GitHub public).
WANDB_API_KEY = ''   # <-- DÁN KEY VÀO GIỮA 2 DẤU NHÁY
assert WANDB_API_KEY, 'Hãy dán WANDB_API_KEY vào ô trên rồi chạy lại cell.'
os.environ['WANDB_API_KEY'] = WANDB_API_KEY
WANDB_PROJECT = f'cv20252-{DS.replace("_", "-")}'   # mỗi dataset 1 project (chung r18/r50)
!wandb login --relogin $WANDB_API_KEY
print('W&B project:', WANDB_PROJECT)

In [ ]:
# 5) TRAIN đủ 12 method + log W&B, log ĐẦY ĐỦ ra cả màn hình lẫn file trên Drive (tee).
#    --skip_existing: Run all lại là train tiếp phần còn dở.
!python run_experiments.py --dataset {DS} --data_root {DATA_ROOT} \
    --methods all --backbones {BB} --seeds 0 1 2 --epochs 30 \
    --holdout_fraction 0.2 --scheduler none --hparams_profile paper \
    --num_workers 2 --batch_size {BATCH_SIZE} --save_dir {SAVE_DIR} --skip_existing \
    --wandb --wandb_project {WANDB_PROJECT} --wandb_group {DS}-{BB}-paper30 \
    --wandb_tags {DS} {BB} paper30 full \
    2>&1 | tee -a {LOG}

In [ ]:
# 6) Tổng hợp kết quả job này -> CSV trên Drive
!python summarize_results.py --metrics_dir {SAVE_DIR} --output_csv {WORK}/outputs/{DS}_{BBTAG}_results.csv
import pandas as pd; pd.read_csv(f'{WORK}/outputs/{DS}_{BBTAG}_results.csv')